# NB01 - The Data-Centric Paradigm
## Background
For most of deep learning's history, researchers treated datasets as fixed and competed on model architectures. Andrew Ng's 2021 advocacy for data-centric AI flipped this: if the data is clean and well-structured, even simple models perform well. Poor data quality cascades through the entire ML pipeline — label noise introduces irreducible error floors, class imbalance skews decision boundaries, and distribution mismatches cause silent deployment failures.

The data-centric movement is now mainstream: companies like Scale AI built billion-dollar businesses on data quality, and modern LLM training invests massively in data curation (deduplication, quality filtering, domain mixing) rather than architectural changes.

## What You'll Learn
- Dataset quality audit: class balance, label consistency, feature statistics
- Detecting duplicate and near-duplicate examples
- Identifying outliers that will harm model performance
- Quality scoring pipeline with actionable recommendations

## Why This Matters
A model trained on clean data almost always outperforms a more complex model trained on noisy data. The dataset audit pipeline here catches the most impactful quality issues before any training begins — saving GPU hours and preventing subtle performance degradation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import Counter

# ── Synthetic dataset with quality issues ─────────────────────────────────
np.random.seed(42)

def make_noisy_dataset(n_samples=1000, n_features=10, n_classes=5,
                        noise_rate=0.1, duplicate_rate=0.05,
                        imbalance_ratio=5.0):
    """Create a dataset with realistic quality issues."""
    # Imbalanced class distribution
    class_weights = np.exp(-np.arange(n_classes) * np.log(imbalance_ratio) / (n_classes-1))
    class_weights /= class_weights.sum()
    labels = np.random.choice(n_classes, size=n_samples, p=class_weights)

    # Generate features with class-conditional means
    class_means = np.random.randn(n_classes, n_features) * 2
    features = np.array([class_means[y] + np.random.randn(n_features) for y in labels])

    # Add label noise
    n_noisy = int(n_samples * noise_rate)
    noisy_idx = np.random.choice(n_samples, n_noisy, replace=False)
    noise_labels = np.random.randint(0, n_classes, n_noisy)
    labels[noisy_idx] = noise_labels

    # Add duplicates
    n_dups = int(n_samples * duplicate_rate)
    dup_idx = np.random.choice(n_samples, n_dups, replace=False)
    features = np.vstack([features, features[dup_idx]])
    labels = np.concatenate([labels, labels[dup_idx]])

    # Add outliers
    n_outliers = 20
    outlier_feats = np.random.randn(n_outliers, n_features) * 10
    outlier_labels = np.random.randint(0, n_classes, n_outliers)
    features = np.vstack([features, outlier_feats])
    labels = np.concatenate([labels, outlier_labels])

    return features, labels

X, y = make_noisy_dataset()
print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes')
print(f'Class distribution: {dict(Counter(y))}')

In [ ]:
# ── Dataset Quality Audit ──────────────────────────────────────────────────
@dataclass
class DataQualityAudit:
    name: str
    issues: List[str] = field(default_factory=list)
    warnings: List[str] = field(default_factory=list)
    stats: Dict = field(default_factory=dict)

    def check_class_balance(self, y: np.ndarray):
        counts = Counter(y)
        total = len(y)
        n_classes = len(counts)
        max_count = max(counts.values())
        min_count = min(counts.values())
        imbalance_ratio = max_count / min_count
        self.stats['class_counts'] = dict(counts)
        self.stats['imbalance_ratio'] = imbalance_ratio
        if imbalance_ratio > 10:
            self.issues.append(f'SEVERE class imbalance: ratio={imbalance_ratio:.1f}')
        elif imbalance_ratio > 3:
            self.warnings.append(f'Moderate class imbalance: ratio={imbalance_ratio:.1f}')

    def check_duplicates(self, X: np.ndarray, tol: float = 1e-6):
        n = len(X)
        # Exact duplicates via hash
        seen = set()
        dup_indices = []
        for i, row in enumerate(X):
            key = tuple(row.round(6))
            if key in seen:
                dup_indices.append(i)
            seen.add(key)
        dup_rate = len(dup_indices) / n
        self.stats['n_duplicates'] = len(dup_indices)
        self.stats['dup_rate'] = dup_rate
        if dup_rate > 0.01:
            self.warnings.append(f'Duplicate samples: {len(dup_indices)} ({100*dup_rate:.1f}%)')
        return dup_indices

    def check_outliers(self, X: np.ndarray, z_thresh: float = 4.0):
        means = X.mean(axis=0)
        stds  = X.std(axis=0) + 1e-10
        z_scores = np.abs((X - means) / stds)
        max_z = z_scores.max(axis=1)
        outlier_mask = max_z > z_thresh
        n_outliers = outlier_mask.sum()
        self.stats['n_outliers'] = int(n_outliers)
        self.stats['outlier_pct'] = float(100 * n_outliers / len(X))
        if n_outliers > 0:
            self.warnings.append(
                f'Outliers: {n_outliers} samples with z-score > {z_thresh}')
        return outlier_mask

    def check_feature_stats(self, X: np.ndarray):
        self.stats['feature_means'] = X.mean(axis=0).tolist()
        self.stats['feature_stds'] = X.std(axis=0).tolist()
        high_var = (np.array(self.stats['feature_stds']) > 10).sum()
        if high_var > 0:
            self.warnings.append(f'{high_var} features with very high variance (std>10)')

    def report(self) -> str:
        lines = [f'=== Data Quality Audit: {self.name} ===']
        lines.append(f'Issues ({len(self.issues)}):')
        for iss in self.issues:
            lines.append(f'  [ISSUE] {iss}')
        lines.append(f'Warnings ({len(self.warnings)}):')
        for w in self.warnings:
            lines.append(f'  [WARN]  {w}')
        quality_score = max(0, 100 - 20*len(self.issues) - 5*len(self.warnings))
        lines.append(f'Quality Score: {quality_score}/100')
        return '\n'.join(lines)

audit = DataQualityAudit('synthetic_dataset')
audit.check_class_balance(y)
dup_idx = audit.check_duplicates(X)
outlier_mask = audit.check_outliers(X)
audit.check_feature_stats(X)
print(audit.report())

In [ ]:
# ── Visualize dataset quality issues ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Class distribution
classes = sorted(audit.stats['class_counts'].keys())
counts = [audit.stats['class_counts'][c] for c in classes]
axes[0].bar(classes, counts, color='steelblue', alpha=0.8)
axes[0].set_title('Class Distribution (Imbalanced)')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Count')
axes[0].axhline(np.mean(counts), color='red', linestyle='--', label='Mean')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# Feature std deviation (outlier detection)
from sklearn.preprocessing import StandardScaler
try:
    from sklearn.preprocessing import StandardScaler
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

means = X.mean(axis=0)
stds  = X.std(axis=0)
z_scores = np.abs((X - means) / (stds + 1e-10)).max(axis=1)
axes[1].hist(z_scores, bins=50, color='steelblue', alpha=0.8)
axes[1].axvline(4.0, color='red', linestyle='--', label='Outlier threshold (z=4)')
axes[1].set_title('Max Z-Score Distribution')
axes[1].set_xlabel('Max z-score per sample'); axes[1].set_ylabel('Count')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

# Feature variance
axes[2].bar(range(X.shape[1]), stds, color='steelblue', alpha=0.8)
axes[2].set_title('Feature Standard Deviations')
axes[2].set_xlabel('Feature index'); axes[2].set_ylabel('Std dev')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{BASE}/s29_01_quality_audit.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Dataset quality audit complete. Duplicates: {len(dup_idx)}, Outliers: {outlier_mask.sum()}')